# MDC — anomaly detection model training (research template)

Semi-supervised pipeline: **Transformer encoder–decoder style autoencoder** on benign windows, **reconstruction error** as anomaly score, **threshold from validation**, **report on test only once**.

**Artifacts:** load `windows.npz` from **Google Drive** (same folder as the export step in `mdc_preprocess_and_dataloader.ipynb`) or from local `data/processed/`.

**Practices included:** reproducible seeds, train/val/test discipline, dropout + weight decay, early stopping (on **benign** val reconstruction), gradient clipping, checkpoint + metrics log, no test peeking until evaluation.


## 0. Environment (Colab)

Run once if needed: `%pip install -q torch torchvision scikit-learn matplotlib`


In [ ]:
import sys
import subprocess

try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch", "scikit-learn", "matplotlib"],
        check=False,
    )

print(f"Running in Colab: {_IN_COLAB}")


Running in Colab: True


## 1. Configuration & reproducibility


In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import numpy as np

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE   = 64
MAX_EPOCHS   = 150
PATIENCE     = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.15
GRAD_CLIP    = 1.0

# ── Model architecture ───────────────────────────────────────────────────────
D_MODEL     = 64
N_HEAD      = 4
N_ENC_LAYERS = 2
DIM_FF      = 128

# ── Run output directory ─────────────────────────────────────────────────────
if _IN_COLAB:
    RUN_DIR = Path("/content/drive/MyDrive/Module4_MDC/runs/last_run")
else:
    RUN_DIR = Path("../outputs/mdc_run")

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)


RUN_DIR: /content/drive/MyDrive/Module4_MDC/runs/last_run


## 2. Mount Drive & load `windows.npz`

**If Drive mounts without asking for login and shows wrong folders:**  
This means Colab is connected to the wrong Google account.  
The cell below uses `force_remount=True` — it will **always ask you to authenticate**.  
Pick the account where your `Module4_MDC/processed/` folder lives.


In [ ]:
import os, json, shutil
import numpy as np
from pathlib import Path

# ── Mount Google Drive ─────────────────────────────────────────────────────
if _IN_COLAB:
    from google.colab import drive

    _mount_path = "/content/drive"

    # Step 1: flush & unmount if mounted
    try:
        drive.flush_and_unmount()
        print("Flushed and unmounted Drive.")
    except Exception:
        pass  # not mounted — that's fine

    # Step 2: remove the leftover directory so mount() starts clean
    if os.path.exists(_mount_path):
        shutil.rmtree(_mount_path)
        print("Removed stale mount directory.")

    # Step 3: fresh mount → triggers Google account auth popup
    os.makedirs(_mount_path, exist_ok=True)
    drive.mount(_mount_path)   # no force_remount needed — directory is empty

    # Step 4: confirm correct account by printing top-level folders
    _myd = Path(_mount_path) / "MyDrive"
    print("\nDrive top-level folders (confirm correct account):")
    for _p in sorted(_myd.iterdir()):
        if _p.is_dir():
            print(f"  {_p.name}/ → {[c.name for c in _p.iterdir()]}")
        else:
            print(f"  {_p.name}")

    DATA_DIR = _myd / "Module4_MDC" / "processed"
else:
    DATA_DIR = Path("../data/processed")

NPZ_PATH = DATA_DIR / "windows.npz"

# ── Search all of Drive if not at default path ─────────────────────────────
def _resolve_npz_path() -> Path:
    if NPZ_PATH.is_file():
        print(f"\n[✓] Found at: {NPZ_PATH}")
        return NPZ_PATH
    if not _IN_COLAB:
        return NPZ_PATH
    print(f"\n[!] Not at default path: {NPZ_PATH}")
    print("    Searching all of My Drive for windows.npz ...")
    myd = Path("/content/drive/MyDrive")
    hits = list(myd.rglob("windows.npz")) if myd.is_dir() else []
    if not hits:
        print("    [✗] windows.npz not found anywhere in My Drive.")
        return NPZ_PATH
    hits.sort(key=lambda p: (
        0 if (p.parent / "export_manifest.json").is_file() else 1,
        len(str(p))
    ))
    print(f"    [✓] Found at: {hits[0]}")
    if len(hits) > 1:
        for h in hits:
            print(f"      {h}")
    return hits[0]

_npz = _resolve_npz_path()

if not _npz.is_file():
    myd = Path("/content/drive/MyDrive")
    print("\n── Full Drive contents ───────────────────────────")
    for p in sorted(myd.iterdir()):
        if p.is_dir():
            print(f"  {p.name}/ → {[c.name for c in p.iterdir()]}")
        else:
            print(f"  {p.name}")
    print("\n── Fix ───────────────────────────────────────────")
    print("1. Re-run this cell and pick the correct Google account.")
    print("2. If windows.npz is missing, run mdc_preprocess_and_dataloader.ipynb first.")
    raise FileNotFoundError(f"windows.npz not found.\nExpected: {NPZ_PATH}")

# ── Load arrays ────────────────────────────────────────────────────────────
z = np.load(_npz)
MANIFEST_PATH = _npz.parent / "export_manifest.json"

X_train = z["X_train"]
X_val   = z["X_val"]
X_test  = z["X_test"]
y_val   = z["y_val"].astype(np.int64)
y_test  = z["y_test"].astype(np.int64)

n_features = X_train.shape[2]
T          = X_train.shape[1]

print(f"\nX_train : {X_train.shape}  dtype={X_train.dtype}")
print(f"X_val   : {X_val.shape}   attack rate={y_val.mean():.4f}")
print(f"X_test  : {X_test.shape}  attack rate={y_test.mean():.4f}")
print(f"T={T} buckets  |  features={n_features}")

if MANIFEST_PATH.is_file():
    print("\nManifest:", json.loads(MANIFEST_PATH.read_text(encoding="utf-8")))

# ── PyTorch device ─────────────────────────────────────────────────────────
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")


Drive not mounted, so nothing to flush and unmount.
Flushed and unmounted Drive.
Removed stale mount directory.
Mounted at /content/drive

Drive top-level folders (confirm correct account):
  214020F_AruranK_SE.pdf
  214020F_Daily_report.pdf
  214020F_Final Report - Aruran Kirupanantha.pdf
  214020F_Monthly Report - april.pdf
  214020F_Monthly Report - february.pdf
  214020F_Monthly Report - july.pdf
  214020F_Monthly Report - june.pdf
  214020F_Monthly Report - march.pdf
  214020F_Monthly Report - may.pdf
  214207K_Independent_Study_Proposal (1).gdoc
  AI-Driven Incident Management Platform — Solution Overview.gdoc
  Aruran K 214020F.jpg
  Aruran K 214020F.pdf
  Aruran K.png
  AruranK_CV.pdf
  AruranK_CV_SE_Inern.pdf
  AruranKirupanantha_SE.pdf
  Budget.pdf
  Classroom/ → ['L3S2 - CLASS 2 B21 Kuppiya']
  Colab Notebooks/ → ['Untitled0.ipynb', 'Feature_Selection_Demo.ipynb', 'Feature_Selection_Demo_Clean.ipynb', 'Feature_Selection_Demo_Clean (1).ipynb', 'Untitled1.ipynb', 'Clustering_2

## 3. PyTorch datasets (benign-only train; full val/test for scoring later)

Shuffle **training** only. **Do not** use test for any decision before §6.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

X_train_t = torch.from_numpy(X_train).float()
X_val_t = torch.from_numpy(X_val).float()
X_test_t = torch.from_numpy(X_test).float()

train_loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)


## 4. Model — attention (Transformer encoder) sequence autoencoder

Input shape `(batch, time, features)`. Output same shape; loss = **MSE** reconstruction. Dropout inside encoder reduces overfitting.


In [ ]:
import torch.nn as nn


class SeqTransformerAE(nn.Module):
    def __init__(
        self,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, n_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.encoder(h)
        return self.output_proj(h)


model = SeqTransformerAE(
    n_features=n_features,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_ENC_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


/tmp/ipykernel_1839/806420903.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Trainable parameters: 87,906


## 5. Train with early stopping (monitor **benign val** reconstruction MSE)

Using only **`y_val == 0`** for the early-stopping metric keeps the model from being “rewarded” for fitting attacks in validation. **Weight decay + dropout + gradient clipping** further reduce overfitting.


In [ ]:
import torch.nn.functional as F
from torch import optim

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)


def batch_mse(pred: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(pred, tgt, reduction="mean")


def eval_benign_val_mse() -> float:
    model.eval()
    xv = X_val_t[y_val == 0].to(device)
    if len(xv) == 0:
        return float("inf")
    with torch.no_grad():
        out = model(xv)
        return float(batch_mse(out, xv).item())


best_state = None
best_val = float("inf")
epochs_no_improve = 0

history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running = 0.0
    n_batches = 0
    for (xb,) in train_loader:
        xb = xb.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon = model(xb)
        loss = batch_mse(recon, xb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        running += float(loss.item())
        n_batches += 1

    train_mse = running / max(n_batches, 1)
    val_mse_benign = eval_benign_val_mse()
    scheduler.step(val_mse_benign)
    history.append({"epoch": epoch, "train_mse": train_mse, "val_mse_benign": val_mse_benign})

    improved = val_mse_benign < best_val - 1e-6
    if improved:
        best_val = val_mse_benign
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:03d}  train_mse={train_mse:.6f}  val_mse_benign={val_mse_benign:.6f}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (best val_mse_benign={best_val:.6f})")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Create RUN_DIR now (Drive is mounted at this point, unlike at config time)
RUN_DIR.mkdir(parents=True, exist_ok=True)

(RUN_DIR / "train_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
torch.save({"model": model.state_dict(), "config": {"T": T, "n_features": n_features, "d_model": D_MODEL}}, RUN_DIR / "checkpoint.pt")
print("Saved:", RUN_DIR / "checkpoint.pt")

epoch 001  train_mse=0.332546  val_mse_benign=0.256733
epoch 005  train_mse=0.019661  val_mse_benign=0.097306
epoch 010  train_mse=0.010056  val_mse_benign=0.064510
epoch 015  train_mse=0.008163  val_mse_benign=0.055395
epoch 020  train_mse=0.007676  val_mse_benign=0.053555
epoch 025  train_mse=0.008278  val_mse_benign=0.052451
epoch 030  train_mse=0.007452  val_mse_benign=0.051446
epoch 035  train_mse=0.007271  val_mse_benign=0.052340
epoch 040  train_mse=0.007147  val_mse_benign=0.051233
epoch 045  train_mse=0.007128  val_mse_benign=0.051811
epoch 050  train_mse=0.007061  val_mse_benign=0.051038
epoch 055  train_mse=0.007059  val_mse_benign=0.050897
epoch 060  train_mse=0.006994  val_mse_benign=0.050921
Early stopping at epoch 63 (best val_mse_benign=0.050648)
Saved: /content/drive/MyDrive/Module4_MDC/runs/last_run/checkpoint.pt


## 6. Anomaly scores & threshold (validation only)

Score = **mean squared error** between input and reconstruction, averaged over time and features. Threshold: **Youden’s J** on **validation** ROC (or you can switch to quantile on benign val only).


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)


@torch.no_grad()
def mse_scores(X: np.ndarray) -> np.ndarray:
    xt = torch.from_numpy(X).float().to(device)
    model.eval()
    out = model(xt)
    err = ((out - xt) ** 2).mean(dim=(1, 2))
    return err.detach().cpu().numpy()


s_val = mse_scores(X_val)
s_test = mse_scores(X_test)

fpr, tpr, thr = roc_curve(y_val, s_val)
# sklearn: len(thr) == len(tpr)-1; pairs (fpr[i+1], tpr[i+1]) use thresholds[i]
j = tpr[1:] - fpr[1:]
ix = int(np.argmax(j))
threshold = float(thr[ix])
print(f"Validation ROC threshold (Youden J): {threshold:.6f}")

y_val_pred = (s_val >= threshold).astype(int)
y_test_pred = (s_test >= threshold).astype(int)

print("Val  ROC-AUC:", roc_auc_score(y_val, s_val))
print("Test ROC-AUC:", roc_auc_score(y_test, s_test))
print("Test PR-AUC :", average_precision_score(y_test, s_test))
print("Test F1     :", f1_score(y_test, y_test_pred))
print("Confusion (test):\n", confusion_matrix(y_test, y_test_pred))

np.savez_compressed(
    RUN_DIR / "scores.npz",
    s_val=s_val,
    s_test=s_test,
    y_val=y_val,
    y_test=y_test,
    threshold=threshold,
)


Validation ROC threshold (Youden J): 0.004714
Val  ROC-AUC: 0.6505632693443772
Test ROC-AUC: 0.6414780230065653
Test PR-AUC : 0.6777231835401896
Test F1     : 0.8300020326580392
Confusion (test):
 [[1817 1852]
 [ 657 6125]]


## 7. Optional: copy checkpoint & scores to Drive

Already under `RUN_DIR` on Colab (`My Drive/Module4_MDC/runs/last_run`). Re-run export after tuning.


In [ ]:
summary = {
    "seed": SEED,
    "threshold": float(threshold),
    "val_roc_auc": float(roc_auc_score(y_val, s_val)),
    "test_roc_auc": float(roc_auc_score(y_test, s_test)),
    "test_pr_auc": float(average_precision_score(y_test, s_test)),
    "test_f1": float(f1_score(y_test, y_test_pred)),
}
(RUN_DIR / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


{
  "seed": 42,
  "threshold": 0.004713826347142458,
  "val_roc_auc": 0.6505632693443772,
  "test_roc_auc": 0.6414780230065653,
  "test_pr_auc": 0.6777231835401896,
  "test_f1": 0.8300020326580392
}
